# Лабораторная работа № 6 — Представления

Вариант 11 «Автостоянка». Задание: [tasks/lab6.md](../../tasks/lab6.md).
Учебные примеры пособия, стр. 121–123, адаптированы на `parking`.

Выполняйте **Run All** после ЛР2 → ЛР3 (ЛР4 допустима, но не обязательна).
Подключение: `PGHOST`, `PGPORT`, `PGUSER`, `PGPASSWORD`, `PGDATABASE`;
по умолчанию `parking` на `127.0.0.1:5432`. Пароль не сохраняется.

Объекты создаются в отдельной схеме `lab6`. Существующие объекты не удаляются.
Повторный запуск обновляет обычные представления и данные материализованных.
Изменения сохраняются только в конце. После прерывания выполните
`conn.rollback()` и повторите запуск с начала.

[Все лабораторные](../../README.md) · [Отчёт](ОТЧЁТ_ЛР6_Автостоянка.md)

## Содержание
1. [Учебные примеры](#training)
2. [Индивидуальные представления](#individual)
3. [Три запроса](#queries)
4. [Проверка обновления](#refresh)


In [2]:
import os
import getpass
import pandas as pd
from IPython.display import display
import psycopg2 as psycopg

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", None)
# При необходимости задайте PGHOST, PGPORT, PGUSER, PGPASSWORD перед запуском.
# Пароль в ноутбуке не сохраняйте: драйвер также поддерживает ~/.pgpass.
DB_NAME = os.getenv("PGDATABASE", "parking")
conn = psycopg.connect(
    host=os.getenv("PGHOST", "127.0.0.1"),
    port=int(os.getenv("PGPORT", "5432")),
    user=os.getenv("PGUSER", getpass.getuser()),
    password=os.getenv("PGPASSWORD") or None,
    dbname=DB_NAME,
)
conn.autocommit = False

def q(sql):
    """Выполнить инструкцию; выборку показать таблицей."""
    try:
        with conn.cursor() as cur:
            cur.execute(sql)
            if cur.description:
                frame = pd.DataFrame(cur.fetchall(), columns=[c[0] for c in cur.description])
                frame.index = range(1, len(frame) + 1)
                display(frame)
                print(f"Строк: {len(frame)}")
            else:
                print(cur.statusmessage)
    except Exception:
        conn.rollback()
        raise

run = q
print("Подключение:", DB_NAME)


Подключение: parking


## Подготовка схемы

Схема изолирует объекты работы от таблиц `public`.

In [3]:
q("""
CREATE SCHEMA IF NOT EXISTS lab6;
""")

CREATE SCHEMA


<a id="training"></a>
## 1. Учебные примеры

### 1.1. Число договоров по клиентам (стр. 121–122)

Аналог `number_of_orders`: вместо заказов считаем договоры.
`COUNT(c.contr_id)` при LEFT JOIN возвращает ноль для клиента без договоров.
Имена столбцов задаются явно: `OR REPLACE` не позволяет переименовывать
существующий столбец добавлением другого псевдонима; для этого есть `ALTER VIEW`.

In [4]:
q("""
CREATE OR REPLACE VIEW lab6.v_client_contract_counts AS
SELECT cl.cl_id, cl.cl_name, COUNT(c.contr_id) AS contract_count
FROM public.clients cl
LEFT JOIN public.contracts c ON c.contr_client_id = cl.cl_id
GROUP BY cl.cl_id, cl.cl_name;
""")

CREATE VIEW


#### Выборка из представления

Сортировка задаётся во внешнем SELECT.

In [5]:
q("""
SELECT * FROM lab6.v_client_contract_counts ORDER BY cl_id;
""")

,cl_id,cl_name,contract_count
1,1,Лебедев Игорь Олегович,2
2,2,Сафина Марина Рустамовна,2
3,3,ООО «Сибтранс»,1
4,4,Круглов Антон Павлович,1
5,5,Нечаева Елена Борисовна,1
6,6,АО «Омскстрой»,1
7,7,Тимофеев Роман Андреевич,1
8,8,Жданова Полина Сергеевна,1


Строк: 8


### 1.2. Суммарная стоимость договоров по категориям (стр. 122)

Аналог `price_view`. Здесь суммируются суммы договоров, а не цены тарифов
с разными периодами. `WITH DATA` сразу сохраняет результат. Повторный запуск
обновляет снимок отдельной командой REFRESH.

In [6]:
q("""
CREATE MATERIALIZED VIEW IF NOT EXISTS lab6.mv_category_amounts AS
SELECT cat.cat_id, cat.cat_name, COUNT(c.contr_id) AS contract_count,
       COALESCE(SUM(c.contr_amount), 0)::numeric(14,2) AS total_amount
FROM public.vehicle_categories cat
LEFT JOIN public.vehicles v ON v.veh_cat_id = cat.cat_id
LEFT JOIN public.contracts c ON c.contr_veh_id = v.veh_id
GROUP BY cat.cat_id, cat.cat_name
WITH DATA;
""")

SELECT 4


#### Обновление снимка

Материализованное представление обновляется явно.

In [7]:
q("""
REFRESH MATERIALIZED VIEW lab6.mv_category_amounts;
""")

REFRESH MATERIALIZED VIEW


#### Выборка по категориям

LEFT JOIN оставляет категории без договоров.

In [8]:
q("""
SELECT cat.cat_name, mv.contract_count, mv.total_amount
FROM public.vehicle_categories cat
LEFT JOIN lab6.mv_category_amounts mv ON mv.cat_id = cat.cat_id
ORDER BY cat.cat_id;
""")

,cat_name,contract_count,total_amount
1,Легковой,4,68975.00
2,Внедорожник,3,54725.00
3,Грузовой,2,189000.00
4,Мотоцикл,1,15000.00


Строк: 4


### 1.3. Клиенты со скидкой (стр. 123)

Аналог `cust_view`. `WITH NO DATA` создаёт ненаполненное представление.
До первого REFRESH чтение вызывает ошибку SQLSTATE 55000.
На повторном запуске IF NOT EXISTS сохраняет уже созданный объект.

In [9]:
q("""
CREATE MATERIALIZED VIEW IF NOT EXISTS lab6.mv_discount_clients AS
SELECT cl_id, cl_name, cl_type, cl_discount
FROM public.clients WHERE cl_discount > 0
WITH NO DATA;
""")

CREATE MATERIALIZED VIEW


In [10]:
with conn.cursor() as cur:
    cur.execute("SELECT ispopulated FROM pg_matviews WHERE schemaname='lab6' AND matviewname='mv_discount_clients'")
    populated = cur.fetchone()[0]
    if not populated:
        cur.execute("SAVEPOINT unpopulated_demo")
        try:
            cur.execute("SELECT * FROM lab6.mv_discount_clients")
        except psycopg.Error as exc:
            state = getattr(exc, "sqlstate", None) or getattr(exc, "pgcode", None)
            cur.execute("ROLLBACK TO SAVEPOINT unpopulated_demo")
            if state != "55000":
                raise
            print("Ожидаемая ошибка 55000: представление ещё не наполнено; необходим REFRESH.")
        else:
            raise AssertionError("Ожидалась ошибка чтения ненаполненного представления")
        finally:
            cur.execute("RELEASE SAVEPOINT unpopulated_demo")
    else:
        print("Повторный запуск: представление уже наполнено.")

Ожидаемая ошибка 55000: представление ещё не наполнено; необходим REFRESH.


#### Наполнение

Теперь представление можно читать.

In [11]:
q("""
REFRESH MATERIALIZED VIEW lab6.mv_discount_clients;
""")

REFRESH MATERIALIZED VIEW


#### Клиенты со скидкой по возрастанию скидки

Код клиента разрешает равенства при сортировке.

In [12]:
q("""
SELECT * FROM lab6.mv_discount_clients ORDER BY cl_discount, cl_id;
""")

,cl_id,cl_name,cl_type,cl_discount
1,1,Лебедев Игорь Олегович,Ф,5
2,5,Нечаева Елена Борисовна,Ф,5
3,3,ООО «Сибтранс»,Ю,10
4,6,АО «Омскстрой»,Ю,15


Строк: 4


<a id="individual"></a>
## 2. Индивидуальные представления

### 2.1. Подробности договоров — обычное представление

Каждая строка — один договор: клиент, автомобиль, место и сумма.
Значение статуса берётся из учебных данных; текущая дата не используется.

In [13]:
q("""
CREATE OR REPLACE VIEW lab6.v_contract_details AS
SELECT c.contr_id, c.contr_number, c.contr_client_id AS cl_id,
       cl.cl_name, cl.cl_type, v.veh_plate, cat.cat_name,
       p.pl_sector, p.pl_number, c.contr_amount, c.contr_status
FROM public.contracts c
JOIN public.clients cl ON cl.cl_id = c.contr_client_id
JOIN public.vehicles v ON v.veh_id = c.contr_veh_id
JOIN public.vehicle_categories cat ON cat.cat_id = v.veh_cat_id
JOIN public.places p ON p.pl_id = c.contr_place_id;
""")

CREATE VIEW


### 2.2. Расчёты по договорам — обычное представление

Платежи сначала группируются по договору, затем присоединяются.
Это сохраняет одну строку на договор и не умножает его сумму на число квитанций.
`balance` — сумма договора минус все платежи; отрицательное значение означает
переплату. Штрафы и возвраты в этой учебной схеме отдельно не учитываются.

In [14]:
q("""
CREATE OR REPLACE VIEW lab6.v_contract_balances AS
SELECT d.*, COALESCE(p.paid_amount, 0)::numeric(14,2) AS paid_amount,
       (d.contr_amount - COALESCE(p.paid_amount, 0))::numeric(14,2) AS balance
FROM lab6.v_contract_details d
LEFT JOIN (
    SELECT pay_contr_id, SUM(pay_sum) AS paid_amount
    FROM public.payments GROUP BY pay_contr_id
) p ON p.pay_contr_id = d.contr_id;
""")

CREATE VIEW


#### Проверка расчётов

Выводятся все договоры, включая полностью оплаченные.

In [15]:
q("""
SELECT contr_number, cl_name, contr_amount, paid_amount, balance FROM lab6.v_contract_balances ORDER BY contr_id;
""")

,contr_number,cl_name,contr_amount,paid_amount,balance
1,Д-2024/001,Лебедев Игорь Олегович,25650.00,25650.00,0.00
2,Д-2024/002,Сафина Марина Рустамовна,27000.00,27000.00,0.00
3,Д-2024/003,ООО «Сибтранс»,97200.00,97200.00,0.00
4,Д-2024/004,Круглов Антон Павлович,33000.00,33000.00,0.00
5,Д-2024/005,Нечаева Елена Борисовна,12825.00,12825.00,0.00
6,Д-2024/006,АО «Омскстрой»,91800.00,45900.00,45900.00
7,Д-2024/007,Тимофеев Роман Андреевич,15000.00,15000.00,0.00
8,Д-2024/008,Жданова Полина Сергеевна,16500.00,5500.00,11000.00
9,Д-2024/009,Лебедев Игорь Олегович,5225.00,10450.00,-5225.00
10,Д-2024/010,Сафина Марина Рустамовна,3500.00,3500.00,0.00


Строк: 10


<a id="queries"></a>
## 3. Три запроса индивидуального задания

### 3.a. Выборка с вложенным запросом

Вывести договоры, сумма которых превышает среднюю сумму всех договоров.
Основной и вложенный запросы обращаются к представлению.

In [16]:
q("""
SELECT contr_number, cl_name, contr_amount
FROM lab6.v_contract_details
WHERE contr_amount > (SELECT AVG(contr_amount) FROM lab6.v_contract_details)
ORDER BY contr_id;
""")

,contr_number,cl_name,contr_amount
1,Д-2024/003,ООО «Сибтранс»,97200.00
2,Д-2024/004,Круглов Антон Павлович,33000.00
3,Д-2024/006,АО «Омскстрой»,91800.00


Строк: 3


### 3.b. Группировка и простая сортировка

Определить число клиентов со скидкой и среднюю скидку по типам клиентов.
Простая сортировка — по одному полю, типу клиента.

In [17]:
q("""
SELECT cl_type, COUNT(*) AS client_count,
       ROUND(AVG(cl_discount), 2) AS average_discount
FROM lab6.mv_discount_clients
GROUP BY cl_type
ORDER BY cl_type;
""")

,cl_type,client_count,average_discount
1,Ф,2,5.00
2,Ю,2,12.50


Строк: 2


### 3.c. Группировка и многоуровневая сортировка

По типу клиента и сектору определить число договоров и суммы начислений,
оплаты и остатка. Сначала сортировать остаток по убыванию, затем начисления
по убыванию, затем тип и сектор по возрастанию.

In [18]:
q("""
SELECT cl_type, pl_sector, COUNT(*) AS contract_count,
       SUM(contr_amount) AS total_amount, SUM(paid_amount) AS paid_amount,
       SUM(balance) AS balance
FROM lab6.v_contract_balances
GROUP BY cl_type, pl_sector
ORDER BY balance DESC, total_amount DESC, cl_type ASC, pl_sector ASC;
""")

,cl_type,pl_sector,contract_count,total_amount,paid_amount,balance
1,Ю,B,2,189000.00,143100.00,45900.00
2,Ф,C,2,29325.00,18325.00,11000.00
3,Ф,B,1,15000.00,15000.00,0.00
4,Ф,A,5,94375.00,99600.00,-5225.00


Строк: 4


<a id="refresh"></a>
## 4. Проверка обычного представления и снимка

В savepoint временно увеличим сумму одного договора на 100. Обычное
представление сразу покажет изменение, материализованное — после REFRESH.
ROLLBACK TO отменяет и изменение договора, и обновление снимка.

In [19]:
with conn.cursor() as cur:
    cur.execute("SELECT MIN(contr_id) FROM public.contracts")
    contract_id = cur.fetchone()[0]
    if contract_id is None:
        raise ValueError("Для демонстрации нужен хотя бы один договор")
    cur.execute("SELECT SUM(contr_amount) FROM lab6.v_contract_details")
    before = cur.fetchone()[0]
    cur.execute("SELECT SUM(total_amount) FROM lab6.mv_category_amounts")
    assert cur.fetchone()[0] == before
    cur.execute("SAVEPOINT refresh_demo")
    try:
        cur.execute("UPDATE public.contracts SET contr_amount=contr_amount+100 WHERE contr_id=%s", (contract_id,))
        cur.execute("SELECT SUM(contr_amount) FROM lab6.v_contract_details")
        live = cur.fetchone()[0]
        cur.execute("SELECT SUM(total_amount) FROM lab6.mv_category_amounts")
        stale = cur.fetchone()[0]
        cur.execute("REFRESH MATERIALIZED VIEW lab6.mv_category_amounts")
        cur.execute("SELECT SUM(total_amount) FROM lab6.mv_category_amounts")
        refreshed = cur.fetchone()[0]
        assert live == before + 100 and stale == before and refreshed == live
        display(pd.DataFrame({"Этап": ["До изменения", "Обычное VIEW после UPDATE", "Снимок до REFRESH", "Снимок после REFRESH"],
                              "Сумма": [before, live, stale, refreshed]}))
    finally:
        cur.execute("ROLLBACK TO SAVEPOINT refresh_demo")
        cur.execute("RELEASE SAVEPOINT refresh_demo")
    cur.execute("SELECT SUM(contr_amount) FROM lab6.v_contract_details")
    assert cur.fetchone()[0] == before
    cur.execute("SELECT SUM(total_amount) FROM lab6.mv_category_amounts")
    assert cur.fetchone()[0] == before
print("Проверка пройдена; исходные суммы и снимок восстановлены.")

,Этап,Сумма
0,До изменения,327700.00
1,Обычное VIEW после UPDATE,327800.00
2,Снимок до REFRESH,327700.00
3,Снимок после REFRESH,327800.00


Проверка пройдена; исходные суммы и снимок восстановлены.


### Состав созданных объектов

Три обычных и два материализованных представления.

In [20]:
q("""
SELECT viewname AS object_name, 'VIEW' AS object_type
FROM pg_views WHERE schemaname = 'lab6'
UNION ALL
SELECT matviewname, 'MATERIALIZED VIEW'
FROM pg_matviews WHERE schemaname = 'lab6'
ORDER BY object_type, object_name;
""")

,object_name,object_type
1,mv_category_amounts,MATERIALIZED VIEW
2,mv_discount_clients,MATERIALIZED VIEW
3,v_client_contract_counts,VIEW
4,v_contract_balances,VIEW
5,v_contract_details,VIEW


Строк: 5


## Завершение

COMMIT сохраняет представления. Изменения демонстрации отменены ранее.

In [21]:
conn.commit()
conn.close()
print("ЛР6 выполнена: представления сохранены.")

ЛР6 выполнена: представления сохранены.
